# 🌸 LILY HUNYUAN 1.5 FAST I2V — KAGGLE

**Goal:** decent-quality image→video on Kaggle without reloading the model for every generation.

Before running: Kaggle → **Settings → Accelerator → GPU** and **Internet → On**. Then use **Run All**.

The default is the official/community Diffusers conversion of **HunyuanVideo‑1.5 480p I2V step-distilled**, using 8–12 denoising steps. Start with **49 frames / 8 steps** to test the session, then move to 73 or 121 frames if it behaves.


In [ ]:
# Cell 1 — install a known Diffusers revision that includes HunyuanVideo 1.5 I2V.
# Do NOT reinstall torch; Kaggle's CUDA-matched torch build is usually the safest one.
import sys, subprocess

packages = [
    "git+https://github.com/huggingface/diffusers.git@c5469b7ceb606edd7ba6570dcd17d38590a18db6",
    "transformers>=4.57.0",
    "accelerate>=1.10.0",
    "huggingface_hub>=0.34.0",
    "safetensors>=0.5.0",
    "sentencepiece",
    "protobuf",
    "bitsandbytes>=0.46.0",
    "kernels>=0.11.0",
    "gradio>=5.45,<7",
    "imageio[ffmpeg]",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
print("✅ Packages installed.")


In [ ]:
# Cell 2 — environment + GPU sanity check
import os, gc, time, uuid, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf-cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import torch
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Kaggle: Settings → Accelerator → GPU, then restart the session.")

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"✅ GPU: {gpu_name}")
print(f"✅ CUDA devices: {gpu_count} | Compute capability: {cc[0]}.{cc[1]} | VRAM: {vram_gb:.1f} GB")

# T4/Turing has poor/no native BF16 tensor-core acceleration, so FP16 is the safer default.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() and cc[0] >= 8 else torch.float16
print("✅ Model dtype:", DTYPE)


In [ ]:
# Cell 3 — load ONCE and keep it alive for every generation
import torch, gc
from diffusers import HunyuanVideo15ImageToVideoPipeline
from diffusers.hooks import apply_group_offloading

MODEL_ID = "hunyuanvideo-community/HunyuanVideo-1.5-Diffusers-480p_i2v_step_distilled"

print("🌸 Loading HunyuanVideo-1.5 step-distilled I2V...")
print("First run downloads the weights. Later generations in THIS Kaggle session reuse the loaded model.")

pipe = HunyuanVideo15ImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)

# Decode video in tiles instead of trying to decode the whole thing at once.
pipe.vae.enable_tiling()
try:
    pipe.vae.enable_slicing()
except Exception:
    pass

cuda = torch.device("cuda:0")
cpu = torch.device("cpu")

# Stream transformer layers through the T4 instead of attempting to park the full 8.3B transformer in VRAM.
# Block-level offload is substantially faster than leaf/sequential offload while keeping memory reasonable.
try:
    pipe.transformer.enable_group_offload(
        onload_device=cuda,
        offload_device=cpu,
        offload_type="block_level",
        num_blocks_per_group=2,
        use_stream=True,
        non_blocking=True,
    )
    print("✅ Transformer: block group-offload enabled")
except Exception as e:
    warnings.warn(f"Block offload was unavailable ({e}). Falling back to leaf offload.")
    pipe.transformer.enable_group_offload(
        onload_device=cuda,
        offload_device=cpu,
        offload_type="leaf_level",
        use_stream=False,
    )

# The Qwen and ByT5 text encoders are also large. Offload their internal blocks.
for name in ("text_encoder", "text_encoder_2"):
    module = getattr(pipe, name, None)
    if module is None:
        continue
    try:
        apply_group_offloading(
            module,
            onload_device=cuda,
            offload_device=cpu,
            offload_type="block_level",
            num_blocks_per_group=4,
            use_stream=True,
            non_blocking=True,
        )
        print(f"✅ {name}: block group-offload enabled")
    except Exception as e:
        warnings.warn(f"{name} block offload unavailable ({e}); using leaf offload.")
        apply_group_offloading(
            module,
            onload_device=cuda,
            offload_device=cpu,
            offload_type="leaf_level",
            use_stream=False,
        )

# These are much smaller and benefit from staying on the GPU.
for name in ("vae", "image_encoder"):
    module = getattr(pipe, name, None)
    if module is not None:
        try:
            module.to(cuda)
        except Exception as e:
            warnings.warn(f"Could not pin {name} to CUDA: {e}")

gc.collect()
torch.cuda.empty_cache()

print("✅ MODEL READY — it will NOT reload between generations.")
print(f"Current allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Cell 4 — iPhone-friendly Gradio studio
import os, time, uuid, gc, traceback
import gradio as gr
import torch
from diffusers.utils import export_to_video

OUTPUT_DIR = "/kaggle/working/lily_hunyuan_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PRESETS = {
    "⚡ TEST — ~2 sec (49 frames)": 49,
    "🌸 BALANCED — ~3 sec (73 frames)": 73,
    "🎬 FULL — ~5 sec (121 frames)": 121,
}

def generate_video(image, prompt, negative_prompt, preset, steps, seed, fps):
    if image is None:
        raise gr.Error("Upload a starting image first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Add a motion / scene prompt.")

    image = image.convert("RGB")
    frames = PRESETS[preset]
    steps = int(steps)
    seed = int(seed)
    fps = int(fps)

    # Step-distilled checkpoint is designed for 8 or 12 steps; 4 is an extra-fast draft option.
    if steps not in (4, 8, 12):
        steps = 8

    out_path = os.path.join(OUTPUT_DIR, f"lily_hy15_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4")
    generator = torch.Generator(device="cuda").manual_seed(seed)

    kwargs = dict(
        image=image,
        prompt=prompt.strip(),
        num_frames=frames,
        num_inference_steps=steps,
        generator=generator,
    )
    if negative_prompt and negative_prompt.strip():
        kwargs["negative_prompt"] = negative_prompt.strip()

    started = time.time()
    try:
        with torch.inference_mode():
            video = pipe(**kwargs).frames[0]
        export_to_video(video, out_path, fps=fps)
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(
            "GPU ran out of memory. Try TEST (49 frames) first. "
            "If this happened immediately after model load, restart the Kaggle session and Run All once."
        )
    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        traceback.print_exc()
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

    seconds = time.time() - started
    del video
    gc.collect()
    torch.cuda.empty_cache()

    status = (
        f"✅ Done in {seconds:.0f}s • {frames} frames • {steps} steps • seed {seed}\n"
        f"Model stayed loaded for the next generation."
    )
    return out_path, status

with gr.Blocks(title="Lily Hunyuan 1.5 Studio", fill_width=True) as demo:
    gr.Markdown(
        "# 🌸 Lily Hunyuan 1.5 Studio\n"
        "Upload an image → describe the movement → generate. "
        "**Start with TEST + 8 steps.** The model stays loaded between generations."
    )

    with gr.Row():
        with gr.Column():
            image_in = gr.Image(type="pil", label="Starting image")
            prompt_in = gr.Textbox(
                label="Motion / scene prompt",
                lines=5,
                placeholder="Example: natural handheld camera, she slowly turns toward the camera, hair moving in a soft breeze..."
            )
            negative_in = gr.Textbox(
                label="Negative prompt (optional)",
                lines=2,
                placeholder="blurry, distorted anatomy, jitter, flicker, low quality"
            )
            preset_in = gr.Dropdown(
                choices=list(PRESETS.keys()),
                value="⚡ TEST — ~2 sec (49 frames)",
                label="Length"
            )
            steps_in = gr.Radio(
                choices=[4, 8, 12],
                value=8,
                label="Steps — 4 fastest / 8 default / 12 best quality"
            )
            with gr.Row():
                seed_in = gr.Number(value=1, precision=0, label="Seed")
                fps_in = gr.Dropdown(choices=[12, 15, 18, 24], value=18, label="Playback FPS")
            go = gr.Button("🌸 GENERATE", variant="primary", size="lg")

        with gr.Column():
            video_out = gr.Video(label="Result", autoplay=True)
            status_out = gr.Textbox(label="Status", interactive=False)

    go.click(
        fn=generate_video,
        inputs=[image_in, prompt_in, negative_in, preset_in, steps_in, seed_in, fps_in],
        outputs=[video_out, status_out],
        concurrency_limit=1,
    )

demo.queue(default_concurrency_limit=1, max_size=8)
demo.launch(share=True, debug=False, show_error=True)
